# Neural Network From Scratch — Day 13 Walkthrough

This notebook explains the experiment pipeline and how to inspect real, reproducible results. It uses the project's NumPy-only model implementation. The full run downloads MNIST on first use and can take several minutes.

## 1. Reproducibility

The script records model initialization seeds, minibatch shuffle seeds, all hyperparameters, data split sizes, histories, and wall-clock time. A deterministic seed makes a run repeatable in the same environment; timing and some low-level numeric behavior can still vary across machines.

In [ ]:
from pathlib import Path
import json, sys
ROOT = Path.cwd()
if not (ROOT / 'nn').exists():
    ROOT = ROOT / 'Neural-Network-From-Scratch'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Project root:', ROOT)
print('Experiment script exists:', (ROOT / 'examples/day13_experiments.py').exists())

## 2. Run the experiment

Run one of these commands from the project root. Full run: `python examples/day13_experiments.py`. Faster pipeline check: `python examples/day13_experiments.py --quick`. The quick run intentionally does not report held-out test accuracy.

In [ ]:
# Uncomment to run the quick pipeline check from this notebook.
# import subprocess
# subprocess.run([sys.executable, str(ROOT / 'examples/day13_experiments.py'), '--quick'], cwd=ROOT, check=True)

## 3. Load and inspect measured results

This cell reads results generated by the script. If the file is missing, run the experiment first. Do not fill missing measurements with estimates.

In [ ]:
results_path = ROOT / 'results/day13_experiments.json'
if results_path.exists():
    results = json.loads(results_path.read_text(encoding='utf-8'))
    print('Split sizes:', results['split_sizes'])
    print('Quick mode:', results['quick_mode'])
    final = results.get('final_experiment')
    if final:
        print('Final configuration:', final['config'])
        print('Epochs completed:', final['epochs_completed'])
        print('Best validation loss:', final['best_validation_loss'])
        print('Final recorded validation accuracy:', final['final_recorded_validation_accuracy'])
        print('Held-out test accuracy:', final.get('test_accuracy', 'not evaluated in this run'))
    for row in results['ablations']:
        print(f"{row['name']}: val_accuracy={row['final_recorded_validation_accuracy']:.4f}, best_val_loss={row['best_validation_loss']:.4f}")
else:
    print('No results file yet. Run examples/day13_experiments.py from the project root.')

## 4. Read the ablation correctly

- **Dropout** tests whether regularization improves validation performance.
- **He vs Xavier** tests hidden-layer initialization; the output layer remains Xavier in both cases.
- **Learning rate** tests a smaller Adam step size.

Compare validation results, not training accuracy alone. These are single-seed experiments and should be treated as exploratory. The held-out test set is reserved for the predeclared final model.

In [ ]:
plot_path = ROOT / 'assets/day13_experiments.png'
print('Plot available:', plot_path.exists(), plot_path)

## 5. Reproduce and extend

Use the same environment and command to reproduce the recorded configuration. For stronger evidence, add multiple seeds per ablation and report mean ± standard deviation. Avoid repeatedly selecting configurations based on the test set; use validation data for model selection and reserve test data for the final estimate.